In [ ]:
import os
import re
import requests
from bs4 import BeautifulSoup

# ✅ 設定儲存資料夾
folder = "C:\\Users\\student\\python_web_scraping-master\\project_gutenberg"
os.makedirs(folder, exist_ok=True)

# ✅ 中文正規表達式（篩選中文，包含全型英數、常用符號）
chinese_pattern = re.compile(r'[\u4e00-\u9fff\u3000-\u303f\uff00-\uffef]+')

# ✅ 判斷是否包含中文（避免純英文書名）
def contains_chinese(text):
    return bool(chinese_pattern.search(text))

# ✅ 中文書籍列表頁
url = "https://www.gutenberg.org/browse/languages/zh"
res = requests.get(url)
soup = BeautifulSoup(res.text, "html.parser")

# ✅ 抓取所有書籍連結
book_links = soup.select("li.pgdbetext > a[href]")
print(f"共發現 {len(book_links)} 本書，開始處理符合條件的書籍...")

valid_count = 0

# ✅ 遍歷所有書籍連結
for i, link in enumerate(book_links):
    raw_title = link.text.strip()

    # 檢查書名是否包含中文（如果是純英文就跳過）
    if not contains_chinese(raw_title):
        print(f"跳過純英文書籍：{raw_title}")
        continue

    book_title = raw_title.replace("/", "_") + ".txt"
    href = "https://www.gutenberg.org" + link['href']
    print(f"({i+1}) 處理：{book_title}")

    # ✅ 進入書籍頁面抓取 .txt.utf-8 連結
    book_res = requests.get(href)
    book_soup = BeautifulSoup(book_res.text, "html.parser")
    text_links = book_soup.select('a[href$=".txt.utf-8"]')

    if not text_links:
        print(f"沒有找到可下載的 .txt 檔：{book_title}")
        continue

    txt_url = text_links[0]['href']
    if txt_url.startswith("//"):
        txt_url = "https:" + txt_url
    elif txt_url.startswith("/"):
        txt_url = "https://www.gutenberg.org" + txt_url

    try:
        txt_data = requests.get(txt_url).text
    except Exception as e:
        print(f"無法下載 {book_title}：{e}")
        continue

    # ✅ 過濾成純中文
    chinese_text = "".join(chinese_pattern.findall(txt_data))

    # ✅ 儲存檔案
    file_path = os.path.join(folder, book_title)
    try:
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(chinese_text)
        print(f"已儲存：{file_path}")
        valid_count += 1
    except Exception as e:
        print(f"無法儲存 {book_title}：{e}")

#print(f"爬蟲完成，共儲存 {valid_count} 本符合條件的中文書籍")


共發現 507 本書，開始處理符合條件的書籍...
(1) 處理：豆棚閒話.txt
已儲存：C:\Users\student\python_web_scraping-master\project_gutenberg\豆棚閒話.txt
(2) 處理：戲中戲.txt
已儲存：C:\Users\student\python_web_scraping-master\project_gutenberg\戲中戲.txt
(3) 處理：比目魚.txt
已儲存：C:\Users\student\python_web_scraping-master\project_gutenberg\比目魚.txt
(4) 處理：比目魚.txt
已儲存：C:\Users\student\python_web_scraping-master\project_gutenberg\比目魚.txt
跳過純英文書籍：Study of Inner Cultivation
(6) 處理：三字經.txt
已儲存：C:\Users\student\python_web_scraping-master\project_gutenberg\三字經.txt
(7) 處理：山水情.txt
已儲存：C:\Users\student\python_web_scraping-master\project_gutenberg\山水情.txt
(8) 處理：山海經.txt
已儲存：C:\Users\student\python_web_scraping-master\project_gutenberg\山海經.txt
(9) 處理：施公案.txt
已儲存：C:\Users\student\python_web_scraping-master\project_gutenberg\施公案.txt
(10) 處理：施公案.txt
已儲存：C:\Users\student\python_web_scraping-master\project_gutenberg\施公案.txt
(11) 處理：易經.txt
已儲存：C:\Users\student\python_web_scraping-master\project_gutenberg\易經.txt
(12) 處理：木蘭奇女傳.txt
已儲存：C:\Users\student\python_w